# 0.8+ Ultimate Audio Deepfake Detection

2026 SOTA 통합: RAPTOR + AASIST3 + Multi-Backbone + Layer-Wise Decision Fusion

In [ ]:
!nvidia-smi

In [ ]:
!pip -q install librosa soundfile transformers accelerate demucs panns-inference onnxruntime-gpu datasets huggingface_hub scikit-learn scipy edge-tts yt-dlp
!apt -qq install -y ffmpeg > /dev/null
print('Done')

In [ ]:
!git clone https://github.com/jogwangjo/da1.git /content/da1
%cd /content/da1
!pwd && ls

In [ ]:
import os
required = ['scripts/build_train_data.py', 'scripts/train_raptor_v2.py', 'scripts/download_librispeech.py', 'scripts/generate_diverse_fakes.py', 'scripts/augment_data.py', 'submit/script_v2.py', 'data/sample_submission.csv']
for f in required:
    status = 'OK' if os.path.exists(f) else 'MISSING'
    print(f'  {status}: {f}')

In [ ]:
import os
from huggingface_hub import snapshot_download

model_dir = 'submit/model'
os.makedirs(model_dir, exist_ok=True)

# DF-Arena 1B
df_dir = f'{model_dir}/df_arena_1b'
if not os.path.exists(f'{df_dir}/pytorch_model.bin'):
    print('Downloading DF-Arena 1B...')
    snapshot_download('Speech-Arena-2025/DF_Arena_1B_V_1', local_dir=df_dir)
    print('Done!')
else:
    print('DF-Arena 1B already exists')

# HTDemucs
htd_dir = f'{model_dir}/htdemucs'
if not os.path.exists(f'{htd_dir}/955717e8-8726e21a.th'):
    print('Downloading HTDemucs...')
    from demucs.pretrained import get_model
    import torch
    m = get_model('htdemucs')
    os.makedirs(htd_dir, exist_ok=True)
    torch.save(m.state_dict(), f'{htd_dir}/955717e8-8726e21a.th')
    print('Done!')
else:
    print('HTDemucs already exists')

# SONICS alpha
sa_dir = f'{model_dir}/sonics-alpha-5s'
if not os.path.exists(f'{sa_dir}/pytorch_model.bin'):
    print('Downloading SONICS alpha...')
    snapshot_download('awsaf49/sonics-spectttra-alpha-5s', local_dir=sa_dir)
    print('Done!')
else:
    print('SONICS alpha already exists')

# SONICS beta
sb_dir = f'{model_dir}/sonics-beta-5s'
if not os.path.exists(f'{sb_dir}/pytorch_model.bin'):
    print('Downloading SONICS beta...')
    snapshot_download('awsaf49/sonics-spectttra-beta-5s', local_dir=sb_dir)
    print('Done!')
else:
    print('SONICS beta already exists')

print('All models done!')

In [ ]:
# LibriSpeech + ASVspoof 2019 + Diverse TTS + Augmentation
!python scripts/download_librispeech.py
!python scripts/build_train_data.py --out train_data --auto-download --max-voice-real 2000 --max-voice-fake 5000 --max-music-real 500 --max-music-fake 2000

In [ ]:
# Ultimate model: mHuBERT + XLS-R dual backbone + Layer-wise fusion
!python scripts/train_raptor_v2.py \
    --train train_data/manifest_train.csv \
    --val train_data/manifest_val.csv \
    --out runs/raptor_v2 \
    --epochs 50 \
    --bs 24 \
    --lr 1e-6 \
    --lr-head 3e-4 \
    --consistency-w 0.25 \
    --p-aug 0.6

In [ ]:
import pandas as pd, os
log_path = 'runs/raptor_v2/log.csv'
if os.path.exists(log_path):
    log = pd.read_csv(log_path)
    print(log.to_string(index=False))
    print(f'\nBest val EER: {log["val_eer"].min():.4f}')
    print(f'Final loss: {log["loss"].iloc[-1]:.4f}')
else:
    print(f'ERROR: {log_path} not found!')

In [ ]:
# Copy trained model + inference with TTA 6
!cp runs/raptor_v2/best.pth submit/model/raptor_best.pth
!python submit/script_v2.py --test-dir data/test --sample-submission data/sample_submission.csv --output output/submission.csv --device cuda --tta 6

In [ ]:
# Check submission
import pandas as pd
sub = pd.read_csv('output/submission.csv')
print(sub.describe())
print(f'\nRows: {len(sub)}')
print(f'FILE_FAKE_PROB range: [{sub["FILE_FAKE_PROB"].min():.4f}, {sub["FILE_FAKE_PROB"].max():.4f}]')

In [ ]:
!python scripts/build_submit_zip.py --script submit/script_v2.py --output submit.zip

In [ ]:
from google.colab import files
files.download('submit.zip')
files.download('output/submission.csv')